# ML Challenge 2026 — Business Entity Resolution

**Workflow:** Clone code from GitHub → install deps → run pipeline → validate output.

**Before running:**
1. Add your dataset to this notebook (the folder containing `train/` and `test/` TSV files)
2. `GITHUB_REPO` is public, so no GitHub secret is required. If you later make the repo
   private, add a Kaggle secret named `GITHUB_TOKEN` with a PAT that has read access — the
   clone cell picks it up automatically.
3. Update `DATASET_SLUG` in the config cell below to match your Kaggle dataset name

In [ ]:
# ── Configuration — edit these before running ─────────────────────────────────
DATASET_SLUG = "ml-challenge-dataset"       # Kaggle dataset name (under /kaggle/input/)
DATASET_OWNER = "balendra221"               # Kaggle dataset owner username
GITHUB_REPO  = "Balendrasingh2210/ml-challenge"
REPO_DIR     = "/kaggle/working/ml-challenge"

import os

# Kaggle mounts datasets at /kaggle/input/{slug} in interactive sessions, but
# at /kaggle/input/datasets/{owner}/{slug} when the notebook is run via the
# API (`kaggle kernels push`) / "Save & Run All". Detect whichever exists.
_candidates = [
    f"/kaggle/input/{DATASET_SLUG}",
    f"/kaggle/input/datasets/{DATASET_OWNER}/{DATASET_SLUG}",
]
DATA_DIR = next((p for p in _candidates if os.path.isdir(p)), _candidates[0])

SRC_DIR      = f"{REPO_DIR}/student_resource/code/business_entity_resolution/src"
UTILS_DIR    = f"{REPO_DIR}/student_resource/utils"
OUTPUT_DIR   = "/kaggle/working/output"

# Pipeline hyperparams
TOP_K        = 60
MAX_PER_KEY  = 80
THRESHOLD    = 0.45   # starting point; auto-tuned during training
TRAIN_S1_CAP = 400_000
WORKERS      = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Config ready. DATA_DIR={DATA_DIR}")

In [ ]:
# ── Clone / pull repo ─────────────────────────────────────────────────────────
# GITHUB_REPO is public, so no token is required. If a GITHUB_TOKEN Kaggle secret
# is present (e.g. for a private repo), it will be used automatically.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    remote = f"https://{token}@github.com/{GITHUB_REPO}.git"
except Exception:
    remote = f"https://github.com/{GITHUB_REPO}.git"

if os.path.exists(f"{REPO_DIR}/.git"):
    print("Repo exists — pulling latest ...")
    !git -C {REPO_DIR} pull
else:
    print("Cloning repo ...")
    !git clone {remote} {REPO_DIR}

# Verify
!git -C {REPO_DIR} log --oneline -3

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q lightgbm rapidfuzz
print("Dependencies installed.")

In [ ]:
# ── Verify dataset paths ──────────────────────────────────────────────────────
expected = [
    f"{DATA_DIR}/train/train_source1.tsv",
    f"{DATA_DIR}/train/train_source2.tsv",
    f"{DATA_DIR}/train/train_source3.tsv",
    f"{DATA_DIR}/train/train_ground_truth.tsv",
    f"{DATA_DIR}/test/test_source1.tsv",
    f"{DATA_DIR}/test/test_source2.tsv",
    f"{DATA_DIR}/test/test_source3.tsv",
]

all_ok = True
for p in expected:
    if os.path.exists(p):
        mb = os.path.getsize(p) / 1024**2
        print(f"  OK  {p}  ({mb:.0f} MB)")
    else:
        print(f"  MISSING  {p}")
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Some dataset files are missing — check DATASET_SLUG in the config cell.")

In [ ]:
# ── Run pipeline ──────────────────────────────────────────────────────────────
import sys

sys.path.insert(0, SRC_DIR)

sys.argv = [
    "pipeline.py",
    "--data-dir",      DATA_DIR,
    "--output-dir",    OUTPUT_DIR,
    "--top-k",         str(TOP_K),
    "--max-per-key",   str(MAX_PER_KEY),
    "--threshold",     str(THRESHOLD),
    "--train-s1-cap",  str(TRAIN_S1_CAP),
    "--workers",       str(WORKERS),
]

# Reload modules in case of a re-run after git pull
for mod in ["normalize", "blocking", "features", "pipeline"]:
    if mod in sys.modules:
        del sys.modules[mod]

from pipeline import main
main()

In [ ]:
# ── Validate submission format ────────────────────────────────────────────────
!python3 {UTILS_DIR}/validate_submission.py \
    --matching  {OUTPUT_DIR}/matching_results.tsv \
    --candidate {OUTPUT_DIR}/candidate_pairs.tsv \
    --test-dir  {DATA_DIR}/test

In [ ]:
# ── Results summary ───────────────────────────────────────────────────────────
import pandas as pd

mr = pd.read_csv(f"{OUTPUT_DIR}/matching_results.tsv", sep='\t', dtype=str).fillna('')
total    = len(mr)
matched  = mr['matched_entity_ids'].str.len().gt(0).sum()
single   = total - matched

print(f"Total Source-1 entities : {total:,}")
print(f"Matched                 : {matched:,}  ({matched/total*100:.1f}%)")
print(f"Singletons (no match)   : {single:,}  ({single/total*100:.1f}%)")
print()

# Show a few matched examples
print("Sample matched rows:")
display(mr[mr['matched_entity_ids'].str.len() > 0].head(10))

In [ ]:
# ── Download output files ─────────────────────────────────────────────────────
# On Kaggle, files saved to /kaggle/working/ appear in the Output tab automatically.
# This cell copies them there and prints their sizes for confirmation.
import shutil

for fname in ["matching_results.tsv", "candidate_pairs.tsv"]:
    src  = f"{OUTPUT_DIR}/{fname}"
    dest = f"/kaggle/working/{fname}"
    if src != dest:
        shutil.copy(src, dest)
    mb = os.path.getsize(dest) / 1024**2
    print(f"  {fname}  ({mb:.1f} MB)  → ready to download from Output tab")